# Job salary prediction, imported Kaggle code
Imported from version 7 on 2026-09-23. All Python code cells are preserved in their original order without changes. Original outputs, images, and narrative cells are omitted. This is third-party reference code, not code authored by this repository's owner.

## Running the notebook

Install dependencies in your Python environment:

```bash
python -m pip install jupyter pandas numpy matplotlib seaborn scikit-learn xgboost
```

Download `job_salary_prediction_dataset.csv` from [the source dataset](https://www.kaggle.com/datasets/nalisha/job-salary-prediction-dataset). On Kaggle, attach that dataset. Locally, edit the `pd.read_csv(...)` path in the second code cell to point to your downloaded CSV. Open this notebook with Jupyter and run the cells in order.

The dataset is not included. The original code uses a 70/30 split and compares polynomial regression, random forest, decision tree, and linear regression. Grid searches are defined but their fitting calls are commented out in the source. Polynomial expansion and random forest training can require substantial memory and time on the full dataset.

This imported version may differ from the analysis in `ds110project.pdf` and the repository README. Its model results have not been rerun or verified during import.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.model_selection import GridSearchCV

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/nalisha/job-salary-prediction-dataset/job_salary_prediction_dataset.csv')
df.head(7)

In [ ]:
print('The DataSize: ', len(df['salary']))

In [ ]:
df.hist(bins=50, figsize=(13,7), color='skyblue')
plt.show()

In [ ]:
ar=df['location'].unique()
plt.style.use('seaborn-v0_8-whitegrid')
ax = df['location'].value_counts().reindex(ar).plot.bar(
    figsize=(7,3), color='#4C72B0'  )

ax.bar_label(ax.containers[0], fmt='%d')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
ar=df['education_level'].unique()
plt.style.use('seaborn-v0_8-whitegrid')
ax = df['education_level'].value_counts().reindex(ar).plot.bar(
    figsize=(8,3), color='#4C72B0'  )

ax.bar_label(ax.containers[0], fmt='%d')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
ar=df['industry'].unique()
ax = df['industry'].value_counts().reindex(ar).plot.bar(
    figsize=(8,4), color='#4C7270'  )

ax.bar_label(ax.containers[0], fmt='%d')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Job title visual
ar=df['job_title'].unique()
ax = df['job_title'].value_counts().reindex(ar).plot.bar(
    figsize=(10,4), color=plt.cm.tab20.colors)

ax.bar_label(ax.containers[0], fmt='%d')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Check Null values  
df.isnull().sum() #Good, There no Missing values

In [ ]:
df.dtypes

In [ ]:
# Hot encodeing the straings columns to boolian columns
col=['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']
dfe = pd.get_dummies(df, columns=col, drop_first=True)

print("The number of Columns After one‑hot encoding:", dfe.shape[1])
dfe.head()

In [ ]:
# # convert strings to Numbers
# df['job_title'] = df['job_title'].astype('category').cat.codes
# df['education_level'] = df['education_level'].astype('category').cat.codes
# df['industry'] = df['industry'].astype('category').cat.codes
# df['company_size'] = df['company_size'].astype('category').cat.codes
# df['location'] = df['location'].astype('category').cat.codes
# df['remote_work'] = df['remote_work'].astype('category').cat.codes

In [ ]:
X=dfe.drop(['salary'],axis=1)
Y=dfe['salary']
xtrain,xtest,ytrain,ytest=train_test_split(X,Y, test_size=0.3,random_state=42)
print(Y.mean())

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(dfe.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
starttime=time.perf_counter()

pol=PolynomialFeatures(degree=2)

xtrainpol=pol.fit_transform(xtrain)
xtestpol = pol.transform(xtest)

modelPR=LinearRegression()
modelPR.fit(xtrainpol, ytrain)

# get the predited answer
train_predictions = modelPR.predict(xtrainpol)
test_predictions =modelPR.predict(xtestpol)


# Cheak the error 
train_mse = mean_squared_error(ytrain, train_predictions)
test_mse = mean_squared_error(ytest, test_predictions)
test_rmse=np.sqrt(test_mse)
test_mae = mean_absolute_error(ytest, test_predictions)
train_r2 = r2_score(ytrain, train_predictions)
test_r2 = r2_score(ytest, test_predictions)



print('Time :=',format(time.perf_counter()-starttime,'.2f'),' seconds')
plt.figure(figsize=(7,3), facecolor='#1E1E2F')
ax = plt.gca()
ax.set_facecolor('#1E1E2F')
plt.axis('off')

plt.title('Polynomial Linear Regression Results', fontsize=14, fontweight='bold', color='white')

style = dict(fontsize=14, ha='left')

plt.text(0.05, 0.85, f"Train MSE: {train_mse:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.75, f"Test MSE: {test_mse:.4f}", color='#FF6B6B', **style)
plt.text(0.05, 0.60, f"RMSE: {test_rmse:.4f}", color='#C77DFF', **style)
plt.text(0.05, 0.50, f"MAE: {test_mae:.4f}", color='#2ED573', **style)
plt.text(0.05, 0.35, f"Train R2: {train_r2:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.25, f"Test R2: {test_r2:.4f}", color='#FF6B6B', **style)

plt.show()


print('\nPolynomial Regression Results')
print('\nThe Predicted results of Unseen Data: \n')
print('---------------------------------------|')
for i in range(20):
    print('Predicted:', format(test_predictions[i],'.2f'), ', Actual:',  format(ytest.iloc[i],'.2f'))

In [ ]:

plt.figure(figsize=(8, 4))

# the actual vs predicted points
plt.scatter(ytrain, train_predictions, alpha=0.6, color='blue', edgecolor='k', label='Predicted vs Actual')

# Plot the perfect fit line (y = x)
min_val = min(ytrain.min(), train_predictions.min())
max_val = max(ytrain.max(), train_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Fit Line')

plt.title('Polynomial Regression: Actual vs. Predicted Salary', fontsize=14)
plt.xlabel('Actual Salary ($)', fontsize=12)
plt.ylabel('Predicted Salary ($)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:


sample_size = min(500, len(ytrain))
np.random.seed(42) 
random_indices = np.random.choice(len(ytrain), size=sample_size, replace=False)

actual_sample = np.array(ytrain)[random_indices]
predict_sample = np.array(train_predictions)[random_indices]

x_axis = np.arange(sample_size)

plt.figure(figsize=(8, 5))

#  ACTUAL salaries 
plt.scatter(x_axis, actual_sample, color='mediumseagreen', alpha=0.7, s=15, label='Actual Salary')

#  PREDICTED salaries 
plt.scatter(x_axis, predict_sample, color='crimson', alpha=0.7, s=15, label='Predicted Salary')


plt.title('Actual vs Predicted Salaries (Random Subsample of 500 Points),  Polynomial', fontsize=14, fontweight='bold')
plt.xlabel('Random Data Point Index', fontsize=12)
plt.ylabel('Salary ($)', fontsize=12)

plt.legend(loc='upper right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {
        'max_depth': [5, 11, 15],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 2],
        'n_estimators': [100, 200, 300]
    },
    cv=5,
    n_jobs=-1
)
bP_RFG={'max_depth': 12, 'min_samples_leaf': 3, 'min_samples_split': 2,'n_estimators':301,'max_features':'sqrt'}
# Remove the # to start the search
# grid.fit(xtrain, ytrain)
# bP_RFG=grid.best_params_
print('Best_parameters of DecisionTreeRegressor: ',bP_RFG)

In [ ]:
starttime=time.perf_counter()

modelR=RandomForestRegressor(
                              **bP_RFG,
                              n_jobs=-1,
                             random_state=42
    )

modelR.fit(xtrain,ytrain)
train_predictions = modelR.predict(xtrain)
test_predictions =modelR.predict(xtest)

train_mse = mean_squared_error(ytrain, train_predictions)
test_mse = mean_squared_error(ytest, test_predictions)
test_rmse=np.sqrt(test_mse)
test_mae = mean_absolute_error(ytest, test_predictions)
train_r2 = r2_score(ytrain, train_predictions)
test_r2 = r2_score(ytest, test_predictions)

print('Time :=',format(time.perf_counter()-starttime,'.2f'),' seconds')
plt.figure(figsize=(7,3), facecolor='#1E1E2F')
ax = plt.gca()

ax.set_facecolor('#1E1E2F')
plt.axis('off')

plt.title('Random Forest Regressor Results', fontsize=14, fontweight='bold', color='white')

style = dict(fontsize=14, ha='left')

plt.text(0.05, 0.85, f"Train MSE: {train_mse:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.75, f"Test MSE: {test_mse:.4f}", color='#FF6B6B', **style)
plt.text(0.05, 0.60, f"RMSE: {test_rmse:.4f}", color='#C77DFF', **style)
plt.text(0.05, 0.50, f"MAE: {test_mae:.4f}", color='#2ED573', **style)
plt.text(0.05, 0.35, f"Train R2: {train_r2:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.25, f"Test R2: {test_r2:.4f}", color='#FF6B6B', **style)

plt.show()

print('\nRandom Forest Regressor Results')
print('\nThe Predicted results of Unseen Data: \n')
print('---------------------------------------|')
for i in range(20):
    print('Predicted:', format(test_predictions[i],'.2f'), '$, Actual:',  format(ytest.iloc[i],'.2f'),'$')

In [ ]:
fi = pd.DataFrame({
    'Feature': xtrain.columns,
    'Importance': modelR.feature_importances_
}).sort_values('Importance').tail(20)

plt.figure(figsize=(8,5))
plt.barh(fi['Feature'], fi['Importance'],
         color=plt.cm.Blues(np.linspace(0.4,1,len(fi))))
plt.title('Random Forest Feature Importance')
plt.show()

In [ ]:
""" """
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {
        'max_depth': [3,5,7,9,11,15,20],
        'min_samples_split': [2,5,10,20],
        'min_samples_leaf': [1,2,5,10]
    },
    cv=5,
    n_jobs=-1
)


bP={'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 2}

# Remove the # to start the search
# grid.fit(xtrain, ytrain)
# bP=grid.best_params_
# {'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 2}
print('Best_parameters of DecisionTreeRegressor: ',bP)

In [ ]:
starttime=time.perf_counter()

modelD=DecisionTreeRegressor( **bP)

modelD.fit(xtrain,ytrain)
train_predictions = modelD.predict(xtrain)
test_predictions =modelD.predict(xtest)

train_mse = mean_squared_error(ytrain, train_predictions)
test_mse = mean_squared_error(ytest, test_predictions)
test_rmse=np.sqrt(test_mse)
test_mae = mean_absolute_error(ytest, test_predictions)
train_r2 = r2_score(ytrain, train_predictions)
test_r2 = r2_score(ytest, test_predictions)

print('Time :=',format(time.perf_counter()-starttime,'.2f'),' seconds')

plt.figure(figsize=(7,3), facecolor='#1E1E2F')
ax = plt.gca()
ax.set_facecolor('#1E1E2F')
plt.axis('off')

plt.title('Decision Tree Regressor Results', fontsize=14, fontweight='bold', color='white')

style = dict(fontsize=14, ha='left')

plt.text(0.05, 0.85, f"Train MSE: {train_mse:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.75, f"Test MSE: {test_mse:.4f}", color='#FF6B6B', **style)
plt.text(0.05, 0.60, f"RMSE: {test_rmse:.4f}", color='#C77DFF', **style)
plt.text(0.05, 0.50, f"MAE: {test_mae:.4f}", color='#2ED573', **style)
plt.text(0.05, 0.35, f"Train R2: {train_r2:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.25, f"Test R2: {test_r2:.4f}", color='#FF6B6B', **style)

plt.show()


print('\nDecision Tree Regressor Results')
print('\nThe Predicted results of Unseen Data: \n')
print('---------------------------------------|')
for i in range(20):
    print('Predicted:', format(test_predictions[i],'.2f'), ', Actual:',  format(ytest.iloc[i],'.2f'))

In [ ]:
fi = pd.DataFrame({
    'Feature': xtrain.columns,
    'Importance': modelD.feature_importances_
}).sort_values('Importance').tail(20)

plt.figure(figsize=(8,5))
plt.barh(fi['Feature'], fi['Importance'],
         color=plt.cm.Blues(np.linspace(0.4,1,len(fi))))

plt.title('Decision Tree Feature Importance')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

starttime=time.perf_counter()

scaler = StandardScaler()
xtrain_scaled = scaler.fit_transform(xtrain)
xtest_scaled = scaler.transform(xtest)


modelLR=LinearRegression()
modelLR.fit(xtrain_scaled,ytrain)
# get the predited answer
train_predictions = modelLR.predict(xtrain_scaled)
test_predictions =modelLR.predict(xtest_scaled)


# Cheak the error 
train_mse = mean_squared_error(ytrain, train_predictions)
test_mse = mean_squared_error(ytest, test_predictions)
test_rmse=np.sqrt(test_mse)
test_mae = mean_absolute_error(ytest, test_predictions)
train_r2 = r2_score(ytrain, train_predictions)
test_r2 = r2_score(ytest, test_predictions)



print('Time :=',format(time.perf_counter()-starttime,'.2f'),' seconds')
plt.figure(figsize=(7,3), facecolor='#1E1E2F')
ax = plt.gca()
ax.set_facecolor('#1E1E2F')
plt.axis('off')

plt.title('Linear Regression Results', fontsize=14, fontweight='bold', color='white')

style = dict(fontsize=14, ha='left')

plt.text(0.05, 0.85, f"Train MSE: {train_mse:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.75, f"Test MSE: {test_mse:.4f}", color='#FF6B6B', **style)
plt.text(0.05, 0.60, f"RMSE: {test_rmse:.4f}", color='#C77DFF', **style)
plt.text(0.05, 0.50, f"MAE: {test_mae:.4f}", color='#2ED573', **style)
plt.text(0.05, 0.35, f"Train R2: {train_r2:.4f}", color='#4DA3FF', **style)
plt.text(0.05, 0.25, f"Test R2: {test_r2:.4f}", color='#FF6B6B', **style)

plt.show()


print('\nLinear Regression Results')
print('\nThe Predicted results of Unseen Data: \n')
print('---------------------------------------|')
for i in range(20):
    print('Predicted:', format(test_predictions[i],'.2f'), ', Actual:',  format(ytest.iloc[i],'.2f'))

In [ ]:

plt.figure(figsize=(8, 5))

# the actual vs predicted points
plt.scatter(ytrain, train_predictions, alpha=0.6, color='blue', edgecolor='k', label='Predicted vs Actual')

# the perfect fit line 
min_val = min(ytrain.min(), train_predictions.min())
max_val = max(ytrain.max(), train_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Fit Line')

plt.title('Linear Regression: Actual vs. Predicted Salary', fontsize=14)
plt.xlabel('Actual Salary ($)', fontsize=12)
plt.ylabel('Predicted Salary ($)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:


sample_size = min(500, len(ytrain))
np.random.seed(42) 
random_indices = np.random.choice(len(ytrain), size=sample_size, replace=False)

actual_sample = np.array(ytrain)[random_indices]
predict_sample = np.array(train_predictions)[random_indices]

x_axis = np.arange(sample_size)

plt.figure(figsize=(8, 5))

#  ACTUAL salaries 
plt.scatter(x_axis, actual_sample, color='mediumseagreen', alpha=0.7, s=15, label='Actual Salary')

#  PREDICTED salaries 
plt.scatter(x_axis, predict_sample, color='crimson', alpha=0.7, s=15, label='Predicted Salary')


plt.title('Actual vs Predicted Salaries (Random Subsample of 500 Points)', fontsize=14, fontweight='bold')
plt.xlabel('Random Data Point Index', fontsize=12)
plt.ylabel('Salary $', fontsize=12)

plt.legend(loc='upper right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()